<a href="https://colab.research.google.com/github/Mouad-Tazka/-ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
        else:
            if r.status_code == 200:
                df = pd.DataFrame(r.json()['daily'])
                if df.empty:
                    raise ValueError("API returned empty weather data")
                return df

            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()
                raise RuntimeError(f"Unexpected status: {r.status_code}")

        if attempt < tries - 1:
            time.sleep(2 ** attempt) # 1s, 2s...

    raise Exception("All retry attempts failed.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

try:
    raw_wx = get_weather('2026-09-05', '2026-10-03')
    wx = raw_wx[raw_wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()

    # Validation check to ensure both requested dates actually arrived
    expected_dates = {'2026-09-05', '2026-10-03'}
    if set(wx['time']) != expected_dates or len(wx) != 2:
        raise ValueError("Missing or duplicate requested dates")

    wx['source'] = 'open-meteo'
except Exception as e:
    print(f"API failed: {e}. Will use fallback.")
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wetter_row = wx.loc[wx['precip_mm'].idxmax()]
drier_row = wx.loc[wx['precip_mm'].idxmin()]

diff = wetter_row['precip_mm'] - drier_row['precip_mm']
wetter_date = wetter_row['date'].strftime('%B %d, %Y')
drier_date = drier_row['date'].strftime('%B %d, %Y')

print(f"{wetter_date} was wetter than {drier_date} by {diff:.1f} mm of precipitation.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime, timezone

print(f"Endpoint: {BASE}")
print("Coordinates: 38.03, -78.51")
print("Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}")
print(f"Data status: {'Fallback sample' if used_fallback else 'Real (open-meteo)'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-12T02:26:50+00:00
Data status: Real (open-meteo)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather fails silently by returning an empty DataFrame, an inner join with our 200 days of sales data will wipe out all of the sales rows, leaving us with zero data overall. If we use a left join, the sales data will be retained but the rain columns will be entirely filled with NaN (null) values. The resulting chart might be completely blank because there are no valid precipitation values to compare against revenue. Depending on how the plotting code handles missing values, the issue could also be hidden or misrepresented. Nobody would catch this initially because the pipeline wouldn't throw an error; it would continue running smoothly while producing a fundamentally broken, useless chart that leads to false business conclusions.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
try:
    # Fetch Orlando
    raw_mco = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
    wx_mco = raw_mco[raw_mco['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    wx_mco['source'] = 'open-meteo'
    wx_mco['city'] = 'Orlando'

    # Prep Charlottesville (using 'wx' built in previous parts)
    wx_cho = wx.copy()

    # Adjust Orlando to match final column names
    wx_mco = wx_mco.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    wx_mco['date'] = pd.to_datetime(wx_mco['date'])

    # Add city to Charlottesville data
    wx_cho['city'] = 'Charlottesville'

    # Combine
    wx_combined = pd.concat([wx_cho, wx_mco])
    wx_combined = wx_combined[['city', 'date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)
    print(wx_combined)

except Exception as e:
    print(f"Stretch API call failed: {e}")

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
